Łączenie składowych EGMS U i E — plik zgodny z istniejącymi analizami
Ten notebook tworzy nowy plik EGMS_Bytom.csv do użycia bez zmian w notebookach:
- EGMS_analysis_pipeline.ipynb
- EGMS_buildings_analysis.ipynb
Pliki wejściowe: EGMS_Bytom_U.csv oraz EGMS_Bytom_E.csv (zmień ścieżki w komórce konfiguracji, jeśli nazwy są inne).

Zasada: rekordy są łączone po pid; do analizy głównej powstają kolumny dat YYYYMMDD jako moduł sqrt(U² + E²). Zachowane są również oryginalne serie składowych w kolumnach YYYYMMDD_U i YYYYMMDD_E, a także prędkości składowych mean_velocity_U i mean_velocity_E.

Ograniczenie interpretacyjne: moduł jest zawsze nieujemny i nie pokazuje, czy punkt osiada, czy się podnosi. Istniejący pipeline potraktuje kolumny dat jako serię modułu przemieszczenia, więc wykryje nietypową wielkość ruchu U–E, a nie kierunek ruchu. Kierunek sprawdzaj w kolumnach U i E. Pełny wektor 3D wymaga również składowej północ–południe.

Pliki źródłowe są tylko odczytywane — kod nie nadpisuje ich.

In [ ]:
import numpy as np
import pandas as pd


In [16]:
FILE_U = "EGMS_Turow_U.csv"
FILE_E = "EGMS_Turow_E.csv"
OUT = "EGMS_Turow_U_E.csv"
 
# "total" - wypadkowe przemieszczenie sqrt(U^2 + E^2) ze znakiem skladowej U
#           (ujemne = osiadanie, tak jak w analizie pionowej)
# "U"     - tylko skladowa pionowa (E dodana jako kolumny pomocnicze)
MODE = "total"
 
# Te same nazwy co w notebooku (obsluga plikow z shapefile'a z ucietymi nazwami)
RENAME = {
    "height_ort": "height_ortho",
    "mean_veloc": "mean_velocity",
    "mean_vel_1": "mean_velocity_std",
    "accelerati": "acceleration",
    "accelera_1": "acceleration_std",
    "seasonalit": "seasonality",
    "seasonal_1": "seasonality_std",
    "gnss_veloc": "gnss_velocity_n",
    "gnss_vel_1": "gnss_velocity_e",
    "gnss_vel_2": "gnss_velocity_u",
}


In [17]:
def load(path):
    d = pd.read_csv(path).rename(columns=RENAME)
    d.columns = [str(c) for c in d.columns]
    d["_key"] = (
        d["easting"].round().astype("int64").astype(str)
        + "_"
        + d["northing"].round().astype("int64").astype(str)
    )
    if d["_key"].duplicated().any():
        raise ValueError(f"{path}: powtarzajace sie pozycje easting/northing")
    return d
 
 
def date_columns(d):
    return [c for c in d.columns if c.isdigit() and len(c) == 8]
 
 
def signed_magnitude(u, e):
    sign = np.where(u < 0, -1.0, 1.0)
    return sign * np.sqrt(u ** 2 + e ** 2)


In [18]:
u = load(FILE_U)
e = load(FILE_E)
 
dates = sorted(set(date_columns(u)) & set(date_columns(e)))
if not dates:
    raise ValueError("Brak wspolnych dat w plikach U i E")
 
# Laczenie po pozycji (pid w U i E moze sie roznic)
m = u.merge(e, on="_key", suffixes=("_U", "_E"), how="inner")
print("Punkty U:", len(u), "| punkty E:", len(e), "| wspolne:", len(m))
print("Wspolne daty:", len(dates), f"({dates[0]} - {dates[-1]})")
 
out = pd.DataFrame({
    "pid": m["pid_U"],
    "easting": m["easting_U"],
    "northing": m["northing_U"],
})
if "height_ortho_U" in m:
    out["height_ortho"] = m["height_ortho_U"]
 
# Parametry zbiorcze
for p in ["mean_velocity", "acceleration", "seasonality"]:
    pu, pe = m[f"{p}_U"].astype(float), m[f"{p}_E"].astype(float)
    out[p] = pu if MODE == "U" else signed_magnitude(pu, pe)
    out[f"{p}_U"] = pu
    out[f"{p}_E"] = pe
    if f"{p}_std_U" in m and f"{p}_std_E" in m:
        out[f"{p}_std"] = np.sqrt(m[f"{p}_std_U"] ** 2 + m[f"{p}_std_E"] ** 2)
 
# Szeregi czasowe - kolumny YYYYMMDD jak w oryginale
U = m[[f"{d}_U" for d in dates]].to_numpy(float)
E = m[[f"{d}_E" for d in dates]].to_numpy(float)
S = U if MODE == "U" else signed_magnitude(U, E)
ts = pd.DataFrame(S, columns=dates)
 
# rmse_ts: RMSE dopasowania trendu liniowego do wynikowego szeregu
dd = pd.to_datetime(dates, format="%Y%m%d")
t = ((dd - dd.min()).days / 365.25).to_numpy()
t_c = t - t.mean()
slope = np.nansum((S - np.nanmean(S, axis=1, keepdims=True)) * t_c, axis=1) / np.sum(t_c ** 2)
fit = np.nanmean(S, axis=1, keepdims=True) + slope[:, None] * t_c
out["rmse_ts"] = np.sqrt(np.nanmean((S - fit) ** 2, axis=1))
 
result = pd.concat([out.reset_index(drop=True), ts], axis=1)
result.to_csv(OUT, index=False)
print(f"Zapisano {OUT}: {len(result)} PS, tryb: {MODE}")


Punkty U: 6268 | punkty E: 6268 | wspolne: 6268
Wspolne daty: 239 (20200308 - 20241206)
Zapisano EGMS_Turow_U_E.csv: 6268 PS, tryb: total
